# 선행, 후행변수_검정
## 1. Granger Causality Test 개요
- 한 시계열 데이터가 다른 시계열 데이터를 예측하는 데 도움을 주는지 확인하는 방법
- '원인'을 밝히는 것이 아니라 예측력의 선행 관계를 검정하는 기법

In [1]:
from hossam import *

from matplotlib import pyplot as plt
from matplotlib import dates
import seaborn as sb
import numpy as np

from pandas import to_datetime, merge, DataFrame
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.stattools import adfuller, grangercausalitytests

📦 아이티윌 이광호 강사가 제작한 라이브러리를 사용중입니다.
📚 자세한 사용 방법은 https://py.hossam.kr 을 참고하세요.
📧 Email: leekh4232@gmail.com
🎬 Youtube: https://www.youtube.com/@hossam-codingclub
📝 Blog: https://blog.hossam.kr/
🔖 Version: 0.5.3

✅ 시각화를 위한 한글 글꼴(NotoSansKR-Regular)이 자동 적용되었습니다.


In [2]:
origin = load_data('apartment_transaction_prices')
print(f"데이터셋 크기: {origin.shape}")
print(f"열 개수: {origin.shape[1]}")
print(f"행 개수: {origin.shape[0]}")
print(origin.info())
origin.head()

2017년 1월 1일부터 2024년 8월 7일까지 건별 아파트 매매가격 데이터 (출처: 국토교통부 실거래가 공개시스템)

필드    설명
------  --------------
APCD    아파트코드
BLAG    년식
FLOR    층
AREA    전용면적(㎡)
CNDT    계약일
TRPR    거래금액(만원)
UPPP    평당단가(만원)

데이터셋 크기: (370096, 7)
열 개수: 7
행 개수: 370096
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 370096 entries, 0 to 370095
Data columns (total 7 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   APCD    370096 non-null  object        
 1   BLAG    370096 non-null  int64         
 2   FLOR    370096 non-null  int64         
 3   AREA    370096 non-null  float64       
 4   CNDT    370096 non-null  datetime64[ns]
 5   TRPR    370096 non-null  int64         
 6   UPPP    370096 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(1)
memory usage: 19.8+ MB
None


,APCD,BLAG,FLOR,AREA,CNDT,TRPR,UPPP
0,A13201209,22,6,84.940,2017-01-01,31700,1231.580
1,A13082703,17,19,59.920,2017-01-01,31800,1751.340
2,A15380403,19,1,84.900,2017-01-01,37000,1438.160
3,A15721010,3,9,84.500,2017-01-01,72500,2831.360
4,A12289501,6,11,84.890,2017-01-01,41800,1624.930


In [3]:
origin1 = load_data('hhbl')
origin1.head()

2017년 1월부터 2024년 7월까지 분기 단위 가계대출 지표 데이터 (출처: 한국은행 경제통계시스템)


,QTRT,HHBL
0,2017-01-01,220704.033
1,2017-04-01,223040.867
2,2017-07-01,230246.200
3,2017-10-01,236541.833
4,2018-01-01,239953.200


## 1. 아파트 평당단가와 가계대출의 관계
### 1. 거래데이터 -> 분기 평균 단가 생성

In [5]:
origin['CNDT'] = to_datetime(origin['CNDT'])
origin1['QTRT'] = to_datetime(origin1['QTRT'])

In [6]:
origin['QTRT'] = origin['CNDT'].dt.to_period('Q').dt.to_timestamp()
price_q = origin.groupby('QTRT')['UPPP'].mean().reset_index()
price_q.head()

,QTRT,UPPP
0,2017-01-01,2385.274
1,2017-04-01,2523.151
2,2017-07-01,2575.211
3,2017-10-01,2871.053
4,2018-01-01,2676.489


### 2. 병합

In [7]:
merged_data = merge(price_q, origin1, on='QTRT', how='inner')
merged_data = merged_data.sort_values('QTRT').dropna()
merged_data.head()

,QTRT,UPPP,HHBL
0,2017-01-01,2385.274,220704.033
1,2017-04-01,2523.151,223040.867
2,2017-07-01,2575.211,230246.200
3,2017-10-01,2871.053,236541.833
4,2018-01-01,2676.489,239953.200


### 3. 스케일링

In [8]:
df = merged_data[['UPPP', 'HHBL']].copy()
scaler = StandardScaler()
sdf = DataFrame(scaler.fit_transform(df), columns = df.columns)
sdf.head()

,UPPP,HHBL
0,-1.653,-1.657
1,-1.515,-1.610
2,-1.463,-1.464
3,-1.167,-1.337
4,-1.361,-1.268


### 4. 차분
#### 1. 정상성 확보될 때 까지의 차수 계산 함수

In [9]:
def get_diff_order(series, alpha = 0.05):
    temp = series.copy()
    d = 0
    while True:
        result = adfuller(temp.dropna())
        if result[1] <= alpha:
            return d
        temp = temp.diff()
        d += 1

#### 2. 함수 적용

In [10]:
d_up = get_diff_order(sdf['UPPP'])
d_hh = get_diff_order(sdf['HHBL'])

print('UPPP 차수:', d_up)
print('HHBL 차수:', d_hh)


UPPP 차수: 1
HHBL 차수: 2


#### 3. 동일 차수 선택

In [11]:
common_d = max(d_up, d_hh)
print('공통 차수 적용:', common_d)

공통 차수 적용: 2


#### 4. 동일 차수로 차분 적용

In [13]:
stationary_df = sdf[['UPPP', 'HHBL']].copy()

for _ in range(common_d):
    stationary_df = stationary_df.diff()

stationary_df = stationary_df.dropna()

stationary_df.head()

,UPPP,HHBL
2,-0.086,0.098
3,0.244,-0.018
4,-0.491,-0.058
5,0.079,0.022
6,0.546,0.008


### 5. Granger Causality Test
#### 1. max_lag = 4 의 의미
- 최대 몇 시점 과거까지 확인할 것인가를 정하는 값
- Granger 검정은 여러 시차(lag)를 각각 따로 검정함
- max_lag = 4->1~4차까지 모두 검사

In [14]:
max_lag = 8            # 분기 데이터: 8로 확대
alpha = 0.05

granger_results = {}

# 1. 가계대출 변화 -> 가격 변화 (y먼저)
test_forward = stationary_df[['UPPP', 'HHBL']].dropna()

min_p_forward = float('inf')
optimal_forward = None
f_stat_forward = None

result_forward = grangercausalitytests(
    test_forward,
    maxlag = max_lag,
    verbose = False
)

for lag, result in result_forward.items():
    p_value = result[0]['ssr_ftest'][1]
    f_stat = result[0]['ssr_ftest'][0]
    if p_value < min_p_forward:
        min_p_forward = p_value
        optimal_forward = lag
        f_stat_forward = f_stat

# 2. 가격 변화 -> 가계대출 변화
test_reverse = stationary_df[['UPPP', 'HHBL']].dropna()

min_p_reverse = float('inf')
optimal_reverse = None
f_stat_reverse = None

result_reverse = grangercausalitytests(
    test_reverse,
    maxlag = max_lag,
    verbose = False
)

for lag, result in result_reverse.items():
    p_value = result[0]['ssr_ftest'][1]
    f_stat = result[0]['ssr_ftest'][0]
    if p_value < min_p_reverse:
        min_p_reverse = p_value
        optimal_reverse = lag
        f_stat_reverse = f_stat

# 3. 방향 판정
if min_p_forward < alpha and min_p_reverse >= alpha:
    category = f'선행 ({optimal_forward}step)'
elif min_p_reverse < alpha and min_p_forward >= alpha:
    category = f'후행 ({optimal_reverse}step)'
if min_p_forward < alpha and min_p_reverse < alpha:
    category = f'양방향 (상호 영향)'
else:
    category = f'X 무관 (선행/후행 관계 없음)'

# 4. 결과 정리
granger_results = {
    'Forward_P_value': [f'{min_p_forward:.6f}'],
    'Forward_F_Stat': [f'{f_stat_forward:.4f}'],
    'Forward_Lag': [optimal_forward],
    'Reverse_P_value': [f'{min_p_reverse:.6f}'],
    'Reverse_F_stat': [f'{f_stat_reverse:.4f}'],
    'Reverse_Lag': [optimal_reverse],
    'Interpretation': [category]
}

granger_results_df = DataFrame(granger_results)
granger_results_df

c:\Users\wodyd\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\wodyd\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


,Forward_P_value,Forward_F_Stat,Forward_Lag,Reverse_P_value,Reverse_F_stat,Reverse_Lag,Interpretation
0,0.011375,4.6989,5,0.011375,4.6989,5,양방향 (상호 영향)
